# URECA: Random Forest Prediction of Fluorophore Photophysical Properties

**Project:** NTU URECA (self-directed), supervised by Prof Liu

This is a single running notebook — every stage of the pipeline (setup, EDA, featurization, modeling, analysis, write-up) lives here as its own section, built up step by step.

---

## What this project is about

We're predicting four photophysical properties of organic fluorophores (chromophores) directly from their molecular structure (SMILES string):

| Target | Column in raw data | Notes |
|---|---|---|
| Absorption max | `Absorption max (nm)` | |
| Emission max | `Emission max (nm)` | |
| Extinction coefficient | `log(e/mol-1 dm3 cm-1)` | already log10-scaled in the raw data |
| Quantum yield (Φ_F) | `Quantum yield` | hardest target — expect this one to underperform |

**Model:** Random Forest is the required model per Prof Liu's 12-week guide (GNN/deep learning is optional future work, only after the classical pipeline is finished and working).

**Key methodological point:** we will split train/test by **Murcko scaffold** (via RDKit), not randomly. A model that does well on a random split but drops on a scaffold split tells us it's not generalizing to genuinely new chemical structures — just memorizing close analogs. That scaffold-vs-random generalization gap is the most promising angle for a possible publication, so we'll track it carefully once we get to modeling.

## Planned sections (all in this notebook)

1. **Setup & data loading** — imports, load raw data, first look
2. EDA — distributions, missingness, solvent effects, correlations
3. Featurization + scaffold split — RDKit descriptors/fingerprints, Murcko scaffold split, random-split baseline
4. RF tuning — hyperparameter search per target
5. Feature importance + hypothesis tests on the scaffold-vs-random gap
6. Error analysis
7. Write-up

## Dataset

We're using the chromophore photophysics database (`chromophore_data.csv`, copied into this folder from the published Scientific Data release). It has **20,836 rows**, one row per (chromophore, solvent) measurement, with columns for the SMILES string, solvent, and the measured photophysical properties above, plus some additional fields (lifetime, FWHM, molecular weight, reference).

Note: many targets have substantial missingness (not every paper reports all four properties), and the same chromophore can appear multiple times across different solvents — both of these matter for how we split and model later, so we're just noting them here and will dig into it properly in the EDA section.

## 1. Imports

Core stack: `pandas`/`numpy` for data handling, `matplotlib`/`seaborn` for plots, `rdkit` for cheminformatics (SMILES parsing, scaffolds, descriptors), `scikit-learn` for the Random Forest models and splitting/metrics utilities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # quiet RDKit's parsing warnings for now

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

print('rdkit, sklearn, pandas all imported OK')

## 2. Load the raw dataset

Loading directly from the CSV in this project folder. We keep a `df_raw` copy untouched so we can always go back to it, and work on `df` going forward.

In [ ]:
DATA_PATH = 'chromophore_data.csv'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = df_raw.copy()

print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 3. Columns we actually care about

Quick rename to shorter, code-friendly names for the columns we'll use repeatedly. We keep the original dataframe intact (`df_raw`) — this renamed view is just for convenience from here on.

In [ ]:
RENAME_MAP = {
    'Chromophore': 'smiles',
    'Solvent': 'solvent',
    'Absorption max (nm)': 'abs_max',
    'Emission max (nm)': 'emi_max',
    'Quantum yield': 'qy',
    'log(e/mol-1 dm3 cm-1)': 'log_ext_coeff',
    'Molecular weight (g mol-1)': 'mol_weight',
}

df = df.rename(columns=RENAME_MAP)

TARGET_COLS = ['abs_max', 'emi_max', 'log_ext_coeff', 'qy']
ID_COLS = ['smiles', 'solvent']

df[ID_COLS + TARGET_COLS].head()

## 4. Sanity checks

Before any real EDA, three quick checks: (a) how much data each target actually has, (b) how many SMILES strings RDKit can parse, (c) how many unique molecules we have vs. total rows (since the same chromophore appears across multiple solvents).

In [ ]:
print('Non-null counts per target (out of {:,} rows):'.format(len(df)))
print(df[TARGET_COLS].notna().sum())
print()
print('Unique SMILES (molecules):', df['smiles'].nunique())
print('Unique solvents:', df['solvent'].nunique())

In [ ]:
def parses_ok(smi):
    try:
        return Chem.MolFromSmiles(smi) is not None
    except Exception:
        return False

parse_mask = df['smiles'].apply(parses_ok)
print(f'SMILES that RDKit can parse: {parse_mask.sum():,} / {len(df):,} ({parse_mask.mean():.1%})')

if (~parse_mask).any():
    print('\nExample unparseable SMILES:')
    display(df.loc[~parse_mask, ['smiles']].head())